# 01 — ドッキング結果処理 / Docking Results Processing

ドッキング出力ファイル（PDBQT / SDF）を読み込み、各ポーズを RDKit Mol に変換します。
続いて受容体に対する ProLIF インタラクションフィンガープリントを計算し、
後続のクラスタリング処理用に実行ごとの SDF + CSV を出力します。

Reads docking output files (PDBQT / SDF), converts each pose to an RDKit Mol,
computes ProLIF interaction fingerprints against the receptor, and exports
per-run SDF + CSV files for downstream clustering.

**以下の CONFIG セルのみ編集してください。 / Edit only the CONFIG cell below.**

In [ ]:
# CONFIG -----------------------------------------------------------------------
CONFIG_PATH = "../notebooks/templates/project_config.toml"  # ← edit this path
# ------------------------------------------------------------------------------

In [ ]:
from pathlib import Path

import pandas as pd
from rdkit import Chem

from docking_analysis import AnalysisConfig, get_reader, ProLIFCalculator

In [ ]:
config = AnalysisConfig.from_toml(CONFIG_PATH)
print(f"Project : {config.project_name}")
print(f"Results : {config.results_dir}")
print(f"Runs    : {len(config.docking_runs)}")
for run in config.docking_runs:
    print(f"  [{run.backend}] {run.result_file.name}  →  protein: {run.protein.name}")

## ポーズの読み込みとインタラクションフィンガープリントの計算
## Load poses and compute interaction fingerprints

In [ ]:
calculator = ProLIFCalculator()
all_residues = [
    r for group in config.interaction_groups for r in group.residues
]

for run in config.docking_runs:
    print(f"\n{'='*60}")
    print(f"Processing: {run.result_file.name}")

    # 1. Read poses
    reader = get_reader(run.backend)
    result = reader.read(run.result_file)
    print(f"  Poses loaded: {len(result.poses)}")

    # 2. Save SDF
    sdf_path = config.results_dir / f"{run.result_file.stem}.sdf"
    writer = Chem.SDWriter(str(sdf_path))
    for mol in result.poses:
        writer.write(mol)
    writer.close()
    print(f"  SDF saved  : {sdf_path}")

    # 3. Build metadata DataFrame
    meta_df = pd.DataFrame({
        "mol_name"     : [m.GetProp("mol_name") for m in result.poses],
        "pose_rank"    : [m.GetProp("pose_rank") for m in result.poses],
        "docking_score": result.scores,
        "source_file"  : [run.result_file.stem] * len(result.poses),
        "protein"      : [run.protein.name] * len(result.poses),
        "backend"      : [run.backend] * len(result.poses),
    })
    if run.n_modes is not None:
        meta_df["n_modes"] = run.n_modes

    # 4. ProLIF interaction fingerprints
    protein_mol = Chem.MolFromPDBFile(str(run.protein.pdb_path), removeHs=False)
    fp_df = calculator.calculate(result.poses, protein_mol, show_progress=True)
    fp_df = calculator.summarize_by_residue(fp_df, all_residues)
    fp_df = ProLIFCalculator.add_hbond_summary_flags(fp_df)

    # 5. Save CSV
    combined = pd.concat(
        [meta_df.reset_index(drop=True), fp_df.reset_index(drop=True)], axis=1
    )
    csv_path = config.results_dir / f"{run.result_file.stem}.csv"
    combined.to_csv(csv_path, index=False)
    print(f"  CSV saved  : {csv_path}  ({combined.shape[0]} poses × {combined.shape[1]} cols)")

## サマリー / Summary

In [ ]:
csv_files = sorted(config.results_dir.glob("*.csv"))
print(f"Generated CSVs: {len(csv_files)}")
summary_rows = []
for f in csv_files:
    df = pd.read_csv(f)
    summary_rows.append({
        "file": f.name,
        "n_poses": len(df),
        "score_min": df["docking_score"].min(),
        "score_max": df["docking_score"].max(),
        "score_mean": df["docking_score"].mean(),
    })
pd.DataFrame(summary_rows).round(3)